In [1]:
import pickle
import json

from boolmore.core.model import Model
from boolmore.io.load import import_phenotypes, check_phenotypes
from boolmore.algo.inference import get_phenotype_prediction
from boolmore.eval.score import get_phenotype_scores
from boolmore.io.export import export_phenotype_results
from boolmore.core.conversions import prime2bnet, prime2rr

In [2]:
json_file = "Tcell_config.json"

INPUT_CACHE = "Tcell_primes_ultrametric.pkl"

INPUT_FILE = "Tcell_data.csv"
OUTPUT_CSV = "Tcell_base_results.csv"

GA_CACHE = "Tcell_primes_ultrametric.pkl"

In [3]:
with open(INPUT_CACHE, "rb") as f:
    primes = pickle.load(f)
print("Loaded primes from cache.")

Loaded primes from cache.


In [4]:
experiments = import_phenotypes(INPUT_FILE)

print(len(experiments))
for d in experiments:
    print(d)
    break

80
PhenotypeExperiment(id=1, perturbation=(), sources=(('APC', 0), ('DLL1', 0), ('IFNA_e', 0), ('IFNG_e', 0), ('IL10_e', 0), ('IL15_e', 0), ('IL1_e', 0), ('IL21_e', 0), ('IL23_e', 0), ('IL25_e', 0), ('IL27_e', 0), ('IL29_e', 0), ('IL2_e', 0), ('IL33_e', 0), ('IL36_e', 0), ('IL4_e', 0), ('IL6_e', 0), ('IL7_e', 0), ('TGFB_e', 0)), phenotype=(('GATA3', 0), ('IFNG', 0), ('IL4', 0), ('TBET', 0)), expected_exists=True, weight=1.0)


In [5]:
check_phenotypes(primes, experiments)

In [6]:
f = open(json_file)
json_dict = json.load(f)

CONSTRAINTS = json_dict["constraints"]

model = Model.import_model(primes, constraints=CONSTRAINTS)

model.check_constraint()

True

In [7]:
print(experiments[1].perturbation)

def _assignment_to_dict(assignment):
    result = {}
    for node, value in assignment:
        result[node] = value
    return result

perturbation = _assignment_to_dict(experiments[1].perturbation)

print(perturbation)

(('IFNG', 1),)
{'IFNG': 1}


In [8]:
predictions = get_phenotype_prediction(primes, experiments, debug=True)

for r in predictions:
    print(r)
    break


Experiment 1
  cache check: 0.000009 s
  get perc_primes: 0.000006 s
  compute max_trap: 0.044524 s

Experiment 2
  cache check: 0.000006 s
  get perc_primes: 0.000006 s
  compute max_trap: 0.023817 s

Experiment 3
  cache check: 0.000007 s
  cache hit (perc_primes)
  get perc_primes: 0.000016 s
  compute max_trap: 0.013782 s

Experiment 4
  cache check: 0.000006 s
  get perc_primes: 0.000006 s
  compute max_trap: 0.021553 s

Experiment 5
  cache check: 0.000022 s
  cache hit (perc_primes)
  get perc_primes: 0.000016 s
  compute max_trap: 0.028805 s

Experiment 6
  cache check: 0.000013 s
  cache hit (perc_primes)
  get perc_primes: 0.000009 s
  compute max_trap: 0.031810 s

Experiment 7
  cache check: 0.000004 s
  get perc_primes: 0.000004 s
  compute max_trap: 0.028160 s

Experiment 8
  cache check: 0.000023 s
  cache hit (perc_primes)
  get perc_primes: 0.000011 s
  compute max_trap: 0.022431 s

Experiment 9
  cache check: 0.000027 s
  cache hit (perc_primes)
  get perc_primes: 0.0

In [9]:
score_items = get_phenotype_scores(experiments, predictions)

print(score_items[0])

EvaluationItemScore(id=1, weight=1.0, agreement=1.0, score=1.0)


In [10]:
export_phenotype_results(experiments, predictions, score_items, OUTPUT_CSV)

id | perturbation           | sources                                                                                                                                                                                                                                                                              | phenotype                                                                                                                                                                        | expected_exists | predicted_exists | agreement | weight | score | found_phenotypes                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [11]:
with open(GA_CACHE, "rb") as f:
    ga_primes = pickle.load(f)
print("Loaded primes from cache.")

print("\n-----comparing with the baseline functions-----")
modified = 0
for node in primes:
    if prime2rr(primes[node])[1] != prime2rr(ga_primes[node])[1]:
        modified += 1
        print("from_ga:" + prime2bnet(node, ga_primes[node]))
        print("from_consensus:" + prime2bnet(node, primes[node]))

print(f"\n{modified} out of {len(primes)} functions differ from the ga results")

Loaded primes from cache.

-----comparing with the baseline functions-----

0 out of 106 functions differ from the ga results
